# Histopathology — MIL aggregator over frozen patch embeddings

**"Light training" from the plan.** Per `docs/12-training-plan.md` and
`docs/04-data-models.md`: no dedicated pathology foundation model was verified
on Hugging Face at planning time, but a strong general vision-language embedding
backbone (MedGemma) is available and usable zero-shot for embeddings. **Only
the small MIL aggregator head needs training** — the embedding backbone stays
frozen.

**What this notebook does:**
1. Loads **PatchCamelyon** (`dpdl-benchmark/patch_camelyon` on Hugging Face) —
   the patch-level CAMELYON derivative used because full gigapixel CAMELYON WSIs
   aren't practical to process on a single Kaggle GPU (docs/04 flags full
   CAMELYON16/17 as "not on HF Hub" for exactly this reason).
2. Groups patches into synthetic "bags" (a stand-in for slide-level bags — a
   real deployment groups patches by their *actual source slide*, not randomly;
   see the note in the last cell) and extracts a frozen embedding per patch.
3. Trains an **attention-based MIL** aggregator (Ilse et al., 2018) that learns
   which patches in a bag matter for the bag-level label — this is what makes
   the model's attention weights usable as a saliency reference later
   (`Finding.saliency_ref`).
4. Evaluates with bag-level AUROC.

## Kaggle setup
- **Add data:** search *Add Data* for **"PatchCamelyon"** — several Kaggle mirrors
  exist (the original PCam Kaggle competition data works too, same images). If
  you'd rather stream it, this notebook also supports pulling `dpdl-benchmark/patch_camelyon`
  directly via `datasets` if you enable Internet.
- **Accelerator:** GPU T4 x2 (embedding extraction is the heavy part; the MIL
  head itself is tiny).
- **Internet:** on (for `pip install` and the Hugging Face model download).

In [ ]:
# Cell 1 — install packages not preinstalled on Kaggle.
!pip install -q datasets transformers accelerate

In [ ]:
# Cell 2 — imports and reproducibility.
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Load PatchCamelyon

Each example is a 96x96 patch with a binary label (tumor tissue present in the
center 32x32 region, or not). We use the Hugging Face `datasets` streaming
loader so this works whether or not you've attached a Kaggle dataset — if you
have attached one instead, swap this cell for `load_dataset("imagefolder", ...)`
pointed at `/kaggle/input/...`.

In [ ]:
# Cell 3 — load PatchCamelyon splits.
# Trimmed to a manageable subset for a single-GPU Kaggle session; raise these
# once you've confirmed the pipeline runs end-to-end.
TRAIN_PATCH_LIMIT = 20_000
VAL_PATCH_LIMIT = 4_000
TEST_PATCH_LIMIT = 4_000

raw_train = load_dataset("dpdl-benchmark/patch_camelyon", split=f"train[:{TRAIN_PATCH_LIMIT}]")
raw_val = load_dataset("dpdl-benchmark/patch_camelyon", split=f"validation[:{VAL_PATCH_LIMIT}]")
raw_test = load_dataset("dpdl-benchmark/patch_camelyon", split=f"test[:{TEST_PATCH_LIMIT}]")
print(raw_train, raw_val, raw_test)

## Frozen embedding backbone

We embed every patch once with a frozen vision encoder and cache the
embeddings — this is the "adapt, don't retrain" principle in practice: the
expensive representation-learning is already done by a pretrained model, we
only pay for a forward pass.

> **Model note:** MedGemma is a large VLM meant to be queried through its
> multimodal chat interface, not as a plain vision-tower embedding extractor —
> pulling clean patch embeddings out of it requires reaching into its vision
> tower internals, which vary by release. For a Kaggle-single-GPU-friendly,
> reliably embeddable alternative we default to a general pretrained ViT here
> (`google/vit-base-patch16-224-in21k`) and note exactly where to swap in
> MedGemma's own vision encoder once you've confirmed its extraction path for
> the exact checkpoint you're using — the training loop below doesn't change
> either way, only `EMBED_MODEL_NAME` and `EMBED_DIM`.

In [ ]:
# Cell 4 — load the frozen embedding backbone.
EMBED_MODEL_NAME = "google/vit-base-patch16-224-in21k"  # swap for MedGemma's vision tower once confirmed

image_processor = AutoImageProcessor.from_pretrained(EMBED_MODEL_NAME)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE).eval()
for parameter in embed_model.parameters():
    parameter.requires_grad_(False)

EMBED_DIM = embed_model.config.hidden_size
print("Embedding dimension:", EMBED_DIM)

In [ ]:
# Cell 5 — embed every patch once, cache to a tensor (avoids re-running the
# backbone every epoch since it's frozen).
@torch.no_grad()
def embed_split(raw_split, batch_size: int = 64) -> tuple[torch.Tensor, torch.Tensor]:
    embeddings, labels = [], []
    for start in range(0, len(raw_split), batch_size):
        batch = raw_split[start : start + batch_size]
        pixel_values = image_processor(images=batch["image"], return_tensors="pt").pixel_values.to(DEVICE)
        pooled = embed_model(pixel_values=pixel_values).last_hidden_state[:, 0, :]  # CLS token
        embeddings.append(pooled.cpu())
        labels.extend(batch["label"])
    return torch.cat(embeddings), torch.tensor(labels, dtype=torch.float32)


train_embeddings, train_labels = embed_split(raw_train)
val_embeddings, val_labels = embed_split(raw_val)
test_embeddings, test_labels = embed_split(raw_test)
print(train_embeddings.shape, val_embeddings.shape, test_embeddings.shape)

## Build bags

MIL needs *bags* of patches with one label per bag, not one label per patch.
PatchCamelyon ships patch-level labels, so here we group patches into
fixed-size synthetic bags (label = "tumor present" if **any** patch in the bag
is positive — the standard MIL assumption) purely to exercise the aggregator.

**In a real deployment, group by actual slide ID** (CAMELYON's real bags are
"all patches from one whole-slide image"), so the model learns real
slide-level heterogeneity, not an artifact of random grouping.

In [ ]:
# Cell 6 — group embeddings into bags.
BAG_SIZE = 20


def make_bags(embeddings: torch.Tensor, labels: torch.Tensor, bag_size: int, seed: int):
    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(len(embeddings), generator=generator)
    bags, bag_labels = [], []
    for start in range(0, len(permutation) - bag_size + 1, bag_size):
        indices = permutation[start : start + bag_size]
        bags.append(embeddings[indices])
        bag_labels.append(float(labels[indices].max()))
    return bags, torch.tensor(bag_labels, dtype=torch.float32)


train_bags, train_bag_labels = make_bags(train_embeddings, train_labels, BAG_SIZE, seed=SEED)
val_bags, val_bag_labels = make_bags(val_embeddings, val_labels, BAG_SIZE, seed=SEED + 1)
test_bags, test_bag_labels = make_bags(test_embeddings, test_labels, BAG_SIZE, seed=SEED + 2)
print(f"bags: train={len(train_bags)} val={len(val_bags)} test={len(test_bags)}")


class BagDataset(Dataset):
    def __init__(self, bags: list[torch.Tensor], bag_labels: torch.Tensor):
        self.bags = bags
        self.bag_labels = bag_labels

    def __len__(self) -> int:
        return len(self.bags)

    def __getitem__(self, index: int):
        return self.bags[index], self.bag_labels[index]


train_loader = DataLoader(BagDataset(train_bags, train_bag_labels), batch_size=1, shuffle=True)
val_loader = DataLoader(BagDataset(val_bags, val_bag_labels), batch_size=1, shuffle=False)
test_loader = DataLoader(BagDataset(test_bags, test_bag_labels), batch_size=1, shuffle=False)

## Attention-based MIL aggregator

For each bag: compute a learned attention weight per patch, pool the patch
embeddings into one bag embedding using those weights, then classify. The
per-patch attention weights are exactly the signal that becomes a
`Finding.saliency_ref`-style overlay later — "this patch is why the model
flagged this slide."

In [ ]:
# Cell 7 — MIL aggregator model.
class AttentionMIL(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, patch_embeddings: torch.Tensor):
        # patch_embeddings: (num_patches, embed_dim)
        attention_logits = self.attention(patch_embeddings).squeeze(-1)  # (num_patches,)
        attention_weights = F.softmax(attention_logits, dim=0)
        bag_embedding = (attention_weights.unsqueeze(-1) * patch_embeddings).sum(dim=0)
        bag_logit = self.classifier(bag_embedding)
        return bag_logit.squeeze(-1), attention_weights


mil_model = AttentionMIL(embed_dim=EMBED_DIM).to(DEVICE)
print(sum(p.numel() for p in mil_model.parameters()), "parameters — tiny, as expected for just the aggregator")

## Training loop

Binary cross-entropy per bag. Bag sizes are equal here (synthetic bags), so no
padding/masking is needed — a real slide-level deployment has variable bag
sizes and would need padding + a mask in the attention softmax.

In [ ]:
# Cell 8 — train the MIL aggregator.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(mil_model.parameters(), lr=1e-4, weight_decay=1e-5)


@torch.no_grad()
def evaluate_mil(loader: DataLoader) -> float:
    mil_model.eval()
    all_true, all_score = [], []
    for patch_embeddings, bag_label in loader:
        patch_embeddings = patch_embeddings.squeeze(0).to(DEVICE)
        logit, _ = mil_model(patch_embeddings)
        all_true.append(bag_label.item())
        all_score.append(torch.sigmoid(logit).item())
    return roc_auc_score(all_true, all_score)


NUM_EPOCHS = 15
best_val_auroc = 0.0
best_state_dict = None

for epoch in range(NUM_EPOCHS):
    mil_model.train()
    running_loss = 0.0
    for patch_embeddings, bag_label in train_loader:
        patch_embeddings = patch_embeddings.squeeze(0).to(DEVICE)
        bag_label = bag_label.to(DEVICE)
        optimizer.zero_grad()
        logit, _ = mil_model(patch_embeddings)
        loss = criterion(logit.unsqueeze(0), bag_label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    val_auroc = evaluate_mil(val_loader)
    print(f"epoch {epoch+1:02d}  train_loss={running_loss/len(train_loader):.4f}  val_auroc={val_auroc:.4f}")

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        best_state_dict = {k: v.cpu().clone() for k, v in mil_model.state_dict().items()}

mil_model.load_state_dict(best_state_dict)
print("Best val AUROC:", best_val_auroc)

In [ ]:
# Cell 9 — held-out test evaluation.
test_auroc = evaluate_mil(test_loader)
print("Test bag-level AUROC:", test_auroc)

## Inspect attention weights (the saliency signal)

This is the piece that maps onto `Finding.saliency_ref` and the cross-modal
grounding novelty candidate from `docs/05-roadmap.md` — the aggregator tells you
*which patches* drove the bag-level call, not just the call itself.

In [ ]:
# Cell 10 — inspect one bag's per-patch attention weights.
sample_patch_embeddings, sample_bag_label = test_bags[0], test_bag_labels[0].item()
with torch.no_grad():
    logit, attention_weights = mil_model(sample_patch_embeddings.to(DEVICE))
print("Bag label:", sample_bag_label, " predicted prob:", torch.sigmoid(logit).item())
print("Per-patch attention weights:", attention_weights.cpu().numpy().round(3))

## Save the checkpoint

In [ ]:
# Cell 11 — save the (tiny) MIL head checkpoint + metadata.
checkpoint_path = "/kaggle/working/pathology_mil_aggregator.pt"
torch.save(mil_model.state_dict(), checkpoint_path)

metadata = {
    "model": "AttentionMIL",
    "embedding_backbone": EMBED_MODEL_NAME,
    "embed_dim": EMBED_DIM,
    "bag_size": BAG_SIZE,
    "test_bag_auroc": test_auroc,
    "trained_on": "PatchCamelyon (dpdl-benchmark/patch_camelyon), synthetic bags",
    "caveat": "Real deployment must group patches by actual slide ID, not random bags.",
}
with open("/kaggle/working/pathology_mil_aggregator.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

print("Saved:", checkpoint_path)
print(json.dumps(metadata, indent=2))

## Next steps (outside this notebook)

1. **Swap synthetic bags for real slide-grouped bags** before trusting any
   metric here as a real signal — this is the single biggest caveat in this
   notebook.
2. Confirm the embedding backbone choice: if MedGemma's own vision tower is
   embeddable for your installed version, prefer it for consistency with the
   CXR vertical's backbone; otherwise a dedicated pathology foundation model
   (re-check Hugging Face — this space moves fast, per docs/04's own caveat)
   may beat a general ViT.
3. Subgroup breakdown + calibration + registry entry, same as every other
   vertical (docs/06, docs/10) — none of that is in this notebook.
4. Wire behind a `SpecialistPort` adapter the same way as the other verticals.